In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df_jobs = pd.read_parquet('/content/drive/MyDrive/NLP-Job-Screener/data/jobs_clean.parquet')


In [ ]:
df_jobs.shape

In [ ]:
df_jobs.head()

In [ ]:
df_jobs['category'].value_counts()

In [ ]:
df_jobs = df_jobs[df_jobs['category'] != 'OTHER']
df_jobs.shape

In [ ]:
df_jobs['title_and_description'] = df_jobs['title'] + " " + df_jobs['description']

In [ ]:
from sklearn.model_selection import train_test_split

X = df_jobs['title_and_description']
y = df_jobs['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit on y_train and transform both y_train and y_test
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)



In [ ]:
from transformers import DistilBertTokenizer

First, we will load the `DistilBertTokenizer` from the `transformers` library, specifying the `distilbert-base-uncased` model. Then, we will use this tokenizer to convert our `X_train` and `X_test` text data into token IDs, attention masks, and other inputs required by the DistilBERT model. This process involves splitting the text into subword units, mapping them to numerical IDs, and padding/truncating sequences to a uniform length suitable for batch processing.

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

In [ ]:
from torch.utils.data import Dataset


class JobDataset(Dataset):
  def __init__(self, texts, labels, tokenizer, max_len=512):
    self.texts = texts
    self.labels = labels
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, idx):
    text = str(self.texts[idx]) # Ensure text is string
    encoding = self.tokenizer(
    text,
    add_special_tokens=True,
    max_length=self.max_len,
    padding='max_length',
    return_attention_mask=True,
    return_tensors='pt',
    truncation=True
)

    return {
        'input_ids': encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'labels': torch.tensor(self.labels[idx], dtype=torch.long)
    }

In [ ]:
train_dataset = JobDataset(X_train.values, y_train_encoded, tokenizer)
test_dataset = JobDataset(X_test.values, y_test_encoded, tokenizer)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
from transformers import DistilBertForSequenceClassification

num_classes = len(label_encoder.classes_)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_classes
)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

In [ ]:
from tqdm import tqdm

epochs = 3
losses = []

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}")
    for batch_idx, batch in loop:
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
      loss = outputs.loss
      epoch_loss += loss.item()

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    loop.set_postfix(loss=loss.item())
    avg_loss = round(epoch_loss / len(train_loader), 4)
    losses.append(avg_loss)
    print(f"Epoch {epoch} | Avg Loss: {avg_loss}")


In [ ]:
model.save_pretrained('/content/drive/MyDrive/NLP-Job-Screener/models/distilbert_classifier')
tokenizer.save_pretrained('/content/drive/MyDrive/NLP-Job-Screener/models/distilbert_classifier')